In [2]:
from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.options import Options
import chromedriver_autoinstaller
from pydantic import BaseModel, AnyUrl, ValidationError
import json


chromedriver_autoinstaller.install()  # Instala o ChromeDriver automaticamente


'c:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\chromedriver_autoinstaller\\140\\chromedriver.exe'

In [3]:
class Imovel(BaseModel):
    title: str
    street: str
    link: str
    price: float
    location: str
    rooms: int
    area: float
    bathrooms: int
    parking: int

In [46]:
# Zapimoveis
def get_important_data_zapimoveis(element):
    # Link do imóvel
    link = element.find_element(By.TAG_NAME, "a").get_attribute("href")
    if link is None:
        link = "None"
    
    # Título do imóvel (ex: "Sala/Conjunto para alugar com 95 m², 1 banheiro, 1 vaga em Jardim, Santo André")
    title = element.find_element(By.TAG_NAME, "a").get_attribute("title")
    
    # Localização (bairro/cidade)
    location = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-location-txt"]').text.strip()
    location = location.split("\n")[1] if "\n" in location else location
    
    # Endereço (rua)
    try:
        street = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-street-txt"]').text.strip()
    except:
        street = ""
    
    # Área
    try:
        area_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-propertyArea-txt"]').text
        area = float(area_txt.split("\n")[-1].replace("m²", "").replace(",", ".").strip())
    except:
        area = 0.0

    # Banheiros
    try:
        bathrooms_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-bathroomQuantity-txt"]').text
        bathrooms = int(bathrooms_txt.split("\n")[-1])
    except:
        bathrooms = 0

    # Vagas
    try:
        rooms_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-bedroomQuantity-txt"]').text
        rooms = int(rooms_txt.split("\n")[-1])
    except:
        rooms = 0

    # Preço
    try:
        price_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-price-txt"] p').text
        price = float(price_txt.split("\n")[-1].replace("R$", "").replace(".", "").replace(",", ".").split("/")[0])
    except:
        price = 0.0
 
    try:
        parking_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-parkingSpacesQuantity-txt"]').text
        parking = int(parking_txt.split("\n")[-1])
    except:
        parking = 0

    try:
        object_imovel = Imovel(
            title=title,
            street=street,
            link=link,
            price=price,
            location=location,
            rooms=rooms,
            area=area,
            bathrooms=bathrooms,
            parking=parking
        )
    except ValidationError as e:
        print(e.errors())
        object_imovel = None
    return object_imovel.model_dump()

In [47]:
# Zapimoveis
# Inicializa o navegador
driver = webdriver.Chrome()

# URL alvo
url = "https://www.zapimoveis.com.br/aluguel/"
driver.get(url)

lista_imoveis = driver.find_elements(By.CSS_SELECTOR, '[data-cy="rp-property-cd"]')
lista = [get_important_data_zapimoveis(links) for links in lista_imoveis]
driver.quit()  # Fecha o navegador

with open("zapimoveis.json", "w", encoding="utf-8") as file:
    json.dump(lista, file, ensure_ascii=False, indent=4)


In [30]:
def get_important_data_ImovelWeb(element):
    # Título do imóvel
    try:
        title = element.find_element(By.CSS_SELECTOR, '[data-qa="POSTING_CARD_DESCRIPTION"]').text
    except:
        title = "None"

    # Link do imóvel
    try:
        link = element.find_element(By.TAG_NAME, 'a').get_attribute("href")
    except:
        link = "None"

    # Localização (bairro/cidade)
    try:
        location = element.find_element(By.CSS_SELECTOR, '[data-qa="POSTING_CARD_LOCATION"]').text.strip()
    except:
        location = "None"

    # Endereço (rua)
    try:
        street = element.find_element(By.CLASS_NAME, "postingLocations-module__location-address-in-listing").text.strip()
    except:
        street = "None"

    # Área, quartos, banheiros, vagas
    area = 0.0
    rooms = 0
    bathrooms = 0
    parking = 0
    try:
        features = element.find_element(By.CSS_SELECTOR, '[data-qa="POSTING_CARD_FEATURES"]')
        spans = features.find_elements(By.TAG_NAME, "span")
        list_values = []
        for span in spans:
            list_values.append(span.text.split()[0])
        area, rooms, bathrooms, parking = list_values[:4]

    except:
        area = 0.0
        rooms = 0
        bathrooms = 0
        parking = 0

    # Preço
    try:
        price_txt = element.find_element(By.CSS_SELECTOR, '[data-qa="POSTING_CARD_PRICE"]').text
        price = float(price_txt.replace("R$", "").replace(".", "").replace(",", ".").split()[0])
    except:
        price = 0.0

    try:
        object_imovel = Imovel(
            title=title,
            street=street,
            link=link,
            price=price,
            location=location,
            rooms=rooms,
            area=area,
            bathrooms=bathrooms,
            parking=parking
        )
    except ValidationError as e:
        print(e.errors())
        object_imovel = None

    return object_imovel.model_dump() if object_imovel else None

In [31]:
# ImovelWeb
# Inicializa o navegador
driver = webdriver.Chrome()

# URL alvo
url = "https://www.imovelweb.com.br/imoveis-venda-santa-catarina.html"
driver.get(url)

lista_imoveis = driver.find_elements(By.CLASS_NAME, 'postingsList-module__card-container')
lista = [get_important_data_ImovelWeb(links) for links in lista_imoveis]
driver.quit()  # Fecha o navegador

with open("imoveisweb.json", "w", encoding="utf-8") as file:
    json.dump(lista, file, ensure_ascii=False, indent=4)